# Phase 4: train Transolver on the SU2 hypersonic dataset

Kaggle orchestrator. Attach the `zeteixeira/su2-hypersonic-sphere-cone`
dataset as input, set Accelerator to GPU T4 x2, Internet on, Run All.

Clones the repo, installs the missing dep, and runs
`scripts/phase4_train_su2.py` once per entry in `INIT_SEEDS` at fixed
`M = 32` (W2 deep-ensemble members). The split seed stays 0, so every
member shares splits and norm stats with the W1 `run_m32` run, which is
ensemble member #1 (init seed 0). Outputs land in
`/kaggle/working/run_m32_s{s}/` and survive the session as notebook
output.

Two runs of 350 epochs fit one 9 h session with margin. Session A:
`INIT_SEEDS = [1, 2]`. Session B: `[3, 4]`.

The dataset version must match the W1 runs (555 cases). If a newer
version was uploaded since, pin the input to the W1 version before
running.

In [ ]:
import os, subprocess, sys, zipfile

REPO = "/kaggle/working/transolver-hypersonic"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/zeteixeira03/transolver-hypersonic.git", REPO],
                   check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops"], check=True)

# find the dataset anywhere under /kaggle/input: kaggle has used both
# /kaggle/input/<slug>/ and /kaggle/input/datasets/<owner>/<slug>/ layouts,
# so walk the tree and score directories by their contents
INPUT = "/kaggle/input"
assert os.path.isdir(INPUT), "no /kaggle/input at all: attach the dataset"
DATA, best = None, -1
for root, dirs, files in os.walk(INPUT, followlinks=True):
    marker = 1000 if ("ledger.db" in files or "su2_cases.zip" in files) else 0
    score = marker + sum(1 for f in files if f.startswith("case_"))
    if score > best:
        best, DATA = score, root
print("selected:", DATA, "score:", best)
assert DATA is not None and best >= 1000, (
    f"no directory with ledger.db or su2_cases.zip under {INPUT}; "
    f"attach the su2 dataset (right panel -> Input -> Add Input)"
)

# the dataset arrives either extracted (case_*.npz + ledger.db at the root)
# or as the single su2_cases.zip if kaggle did not auto-extract the archive
if not os.path.isfile(f"{DATA}/ledger.db"):
    zsrc = f"{DATA}/su2_cases.zip"
    extracted = "/kaggle/working/su2_data"
    if not os.path.isfile(f"{extracted}/ledger.db"):
        os.makedirs(extracted, exist_ok=True)
        with zipfile.ZipFile(zsrc) as z:
            z.extractall(extracted)
    DATA = extracted

n_cases = len([d for d in os.listdir(DATA) if d.startswith("case_")])
print("cases:", n_cases)
assert n_cases > 100, f"only {n_cases} case files under {DATA}; wrong dataset attached?"

In [ ]:
import subprocess

INIT_SEEDS = [1, 2]         # session A; set to [3, 4] for session B
M = 32                      # fixed W2 ensemble slice count (W1 in-distribution optimum)
EPOCHS = 350

def run(cmd):
    # stream child output into the cell; ! magics can't loop with error checks
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"run failed: {' '.join(cmd)}")

for s in INIT_SEEDS:
    print(f"===== init_seed={s} =====", flush=True)
    run([sys.executable, "scripts/phase4_train_su2.py",
         "--workdir", DATA,
         "--out", f"/kaggle/working/run_m{M}_s{s}",
         "--slice-num", str(M),
         "--epochs", str(EPOCHS),
         "--val-every", "10", "--seed", "0",
         "--init-seed", str(s)])

In [ ]:
import glob, json

for path in sorted(glob.glob("/kaggle/working/run_m*/final_eval.json")):
    with open(path) as f:
        final = json.load(f)
    print(f"===== {path} (slice_num={final['args']['slice_num']}, "
          f"init_seed={final['args'].get('init_seed')}) =====")
    print(json.dumps(final["final"], indent=2))
    print("splits:", final["splits"])